<a href="https://colab.research.google.com/github/ggalarza1/accounting/blob/main/utxo_accounting_extractor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#!/usr/bin/env python3
"""
UTXO ACCOUNTING EXTRACTOR
Extracts UTXO data from Bitcoin wallets with cost basis tracking
Essential for proper FIFO/LIFO/Specific ID accounting
"""

import requests
import json
from datetime import datetime
from typing import List, Dict, Optional, Tuple
from collections import defaultdict


class UTXOAccountingExtractor:
    """
    Extract UTXO data with full accounting context
    """

    def __init__(self, api_source: str = "mempool"):
        self.api_source = api_source
        self.base_url = self._get_api_url()
        self.session = requests.Session()

    def _get_api_url(self) -> str:
        """Get appropriate API endpoint"""
        if self.api_source == "mempool":
            return "https://mempool.space/api"
        elif self.api_source == "blockstream":
            return "https://blockstream.info/api"
        return "https://mempool.space/api"

    def get_address_utxos(self, address: str) -> List[Dict]:
        """
        Get all unspent transaction outputs for an address
        These are the "bills" in your Bitcoin wallet
        """
        url = f"{self.base_url}/address/{address}/utxo"

        try:
            response = self.session.get(url, timeout=30)
            response.raise_for_status()
            utxos = response.json()

            # Enrich each UTXO with accounting data
            enriched_utxos = []
            for utxo in utxos:
                enriched = self._enrich_utxo(utxo, address)
                enriched_utxos.append(enriched)

            return enriched_utxos

        except requests.exceptions.RequestException as e:
            print(f"Error fetching UTXOs: {e}")
            return []

    def _enrich_utxo(self, utxo: Dict, address: str) -> Dict:
        """
        Add accounting-relevant data to raw UTXO
        """
        # Get the transaction that created this UTXO
        tx_id = utxo.get('txid')
        vout = utxo.get('vout')

        # Fetch transaction details to get timestamp
        tx_details = self._get_transaction_details(tx_id)

        # Calculate value in BTC (from satoshis)
        value_sats = utxo.get('value', 0)
        value_btc = value_sats / 100_000_000

        # Get block time for cost basis date
        block_time = tx_details.get('status', {}).get('block_time')
        acquisition_date = datetime.fromtimestamp(block_time).isoformat() if block_time else None

        # Fetch historical price for cost basis
        cost_basis_usd = None
        if acquisition_date:
            cost_basis_usd = self._get_historical_price(acquisition_date[:10], value_btc)

        return {
            # Technical UTXO data
            'tx_id': tx_id,
            'vout': vout,
            'address': address,
            'value_sats': value_sats,
            'value_btc': value_btc,

            # Accounting data
            'acquisition_date': acquisition_date,
            'confirmations': utxo.get('status', {}).get('confirmed', False),
            'block_height': utxo.get('status', {}).get('block_height'),

            # Cost basis (CRITICAL for accounting)
            'cost_basis_usd': cost_basis_usd,
            'cost_basis_per_btc': cost_basis_usd / value_btc if cost_basis_usd and value_btc > 0 else None,

            # Current value (for unrealized gain/loss)
            'current_price_usd': self._get_current_price(),
            'current_value_usd': value_btc * self._get_current_price() if self._get_current_price() else None,

            # Gain/loss calculation
            'unrealized_gain_usd': (value_btc * self._get_current_price() - cost_basis_usd)
                                  if cost_basis_usd and self._get_current_price() else None
        }

    def _get_transaction_details(self, tx_id: str) -> Dict:
        """Fetch transaction details including timestamp"""
        url = f"{self.base_url}/tx/{tx_id}"

        try:
            response = self.session.get(url, timeout=30)
            response.raise_for_status()
            return response.json()
        except:
            return {}

    def _get_historical_price(self, date: str, btc_amount: float) -> Optional[float]:
        """
        Get historical USD price for Bitcoin on specific date
        Uses CoinGecko API (free tier)
        """
        try:
            # Convert date format
            date_obj = datetime.strptime(date, '%Y-%m-%d')
            api_date = date_obj.strftime('%d-%m-%Y')

            url = f"https://api.coingecko.com/api/v3/coins/bitcoin/history"
            params = {'date': api_date, 'localization': 'false'}

            response = requests.get(url, params=params, timeout=30)
            if response.status_code == 429:
                print(f"Rate limit hit for {date}, waiting...")
                import time
                time.sleep(2)
                return self._get_historical_price(date, btc_amount)

            response.raise_for_status()
            data = response.json()

            price = data.get('market_data', {}).get('current_price', {}).get('usd')
            if price:
                return btc_amount * price
            return None

        except Exception as e:
            print(f"Error fetching price for {date}: {e}")
            return None

    def _get_current_price(self) -> Optional[float]:
        """Get current Bitcoin price in USD"""
        try:
            url = "https://api.coingecko.com/api/v3/simple/price"
            params = {'ids': 'bitcoin', 'vs_currencies': 'usd'}
            response = requests.get(url, params=params, timeout=10)
            return response.json().get('bitcoin', {}).get('usd')
        except:
            return None

    def calculate_cost_basis_methods(self, utxos: List[Dict]) -> Dict:
        """
        Calculate cost basis under different accounting methods
        """
        if not utxos:
            return {}

        # Sort by acquisition date for FIFO
        sorted_by_date = sorted(utxos, key=lambda x: x.get('acquisition_date') or '9999-12-31')

        # Sort by cost basis per BTC for LIFO (highest cost first)
        sorted_by_cost = sorted(utxos, key=lambda x: x.get('cost_basis_per_btc') or 0, reverse=True)

        total_btc = sum(u['value_btc'] for u in utxos)

        # FIFO (First In, First Out)
        fifo_data = {
            'cost_basis_total': sum(u.get('cost_basis_usd', 0) or 0 for u in sorted_by_date),
            'cost_basis_per_btc': sum(u.get('cost_basis_usd', 0) or 0 for u in sorted_by_date) / total_btc if total_btc > 0 else 0,
            'method': 'FIFO',
            'utxo_order': [u['tx_id'][:8] for u in sorted_by_date[:5]]  # Show first 5
        }

        # LIFO (Last In, First Out)
        lifo_data = {
            'cost_basis_total': sum(u.get('cost_basis_usd', 0) or 0 for u in sorted_by_date[::-1]),
            'cost_basis_per_btc': sum(u.get('cost_basis_usd', 0) or 0 for u in sorted_by_date[::-1]) / total_btc if total_btc > 0 else 0,
            'method': 'LIFO',
            'utxo_order': [u['tx_id'][:8] for u in sorted_by_date[-5:]]  # Show last 5
        }

        # HIFO (Highest In, First Out) - tax optimization
        hifo_data = {
            'cost_basis_total': sum(u.get('cost_basis_usd', 0) or 0 for u in sorted_by_cost),
            'cost_basis_per_btc': sum(u.get('cost_basis_usd', 0) or 0 for u in sorted_by_cost) / total_btc if total_btc > 0 else 0,
            'method': 'HIFO',
            'utxo_order': [u['tx_id'][:8] for u in sorted_by_cost[:5]]  # Show highest cost first
        }

        return {
            'FIFO': fifo_data,
            'LIFO': lifo_data,
            'HIFO': hifo_data,
            'total_utxos': len(utxos),
            'total_btc': total_btc
        }

    def simulate_spend(self, utxos: List[Dict], spend_amount_btc: float, method: str = 'FIFO') -> Dict:
        """
        Simulate spending Bitcoin using different cost basis methods
        Shows tax implications
        """
        if not utxos:
            return {'error': 'No UTXOs available'}

        # Sort based on method
        if method == 'FIFO':
            sorted_utxos = sorted(utxos, key=lambda x: x.get('acquisition_date') or '9999-12-31')
        elif method == 'LIFO':
            sorted_utxos = sorted(utxos, key=lambda x: x.get('acquisition_date') or '0000-01-01', reverse=True)
        elif method == 'HIFO':
            sorted_utxos = sorted(utxos, key=lambda x: x.get('cost_basis_per_btc') or 0, reverse=True)
        else:
            return {'error': f'Unknown method: {method}'}

        # Select UTXOs to spend
        selected = []
        remaining_to_spend = spend_amount_btc

        for utxo in sorted_utxos:
            if remaining_to_spend <= 0:
                break

            utxo_value = utxo['value_btc']
            selected.append(utxo)
            remaining_to_spend -= utxo_value

        if remaining_to_spend > 0:
            return {'error': f'Insufficient funds. Need {spend_amount_btc} BTC, have {sum(u["value_btc"] for u in utxos):.8f} BTC'}

        # Calculate gain/loss
        total_cost_basis = sum(u.get('cost_basis_usd', 0) or 0 for u in selected)
        current_price = self._get_current_price() or 0
        proceeds = spend_amount_btc * current_price

        # Change calculation (if any)
        change_btc = sum(u['value_btc'] for u in selected) - spend_amount_btc
        change_cost_basis = 0
        if change_btc > 0 and len(selected) > 0:
            # Allocate cost basis proportionally
            total_selected_value = sum(u['value_btc'] for u in selected)
            change_ratio = change_btc / total_selected_value
            change_cost_basis = total_cost_basis * change_ratio

        gain_loss = proceeds - (total_cost_basis - change_cost_basis)

        return {
            'spend_amount_btc': spend_amount_btc,
            'method_used': method,
            'utxos_spent': len(selected),
            'selected_utxos': [
                {
                    'tx_id': u['tx_id'][:16] + '...',
                    'amount_btc': u['value_btc'],
                    'acquisition': u['acquisition_date'][:10] if u['acquisition_date'] else 'Unknown',
                    'cost_basis': u.get('cost_basis_usd', 0)
                }
                for u in selected
            ],
            'total_cost_basis': total_cost_basis - change_cost_basis,
            'proceeds_usd': proceeds,
            'gain_loss_usd': gain_loss,
            'taxable_event': gain_loss != 0,
            'change_btc': change_btc,
            'change_cost_basis': change_cost_basis
        }

    def export_accounting_report(self, utxos: List[Dict], filename: str = 'utxo_accounting_report.csv'):
        """Export UTXO data in accounting-friendly format"""
        import csv

        fieldnames = [
            'tx_id', 'vout', 'address', 'value_btc', 'acquisition_date',
            'cost_basis_usd', 'current_value_usd', 'unrealized_gain_usd',
            'confirmations', 'block_height'
        ]

        with open(filename, 'w', newline='') as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()

            for utxo in utxos:
                row = {k: utxo.get(k, '') for k in fieldnames}
                writer.writerow(row)

        print(f"✅ UTXO accounting report exported to {filename}")


def demo():
    """Demo the UTXO accounting extractor"""
    print("=" * 80)
    print("UTXO ACCOUNTING EXTRACTOR")
    print("Bitcoin Cost Basis Tracking for Tax Compliance")
    print("=" * 80)

    extractor = UTXOAccountingExtractor()

    # Example: Use a known Bitcoin address (Satoshi's genesis address)
    demo_address = "1A1zP1eP5QGefi2DMPTfTL5SLmv7DivfNa"

    print(f"\n🔍 Analyzing UTXOs for address: {demo_address}")
    print("   (Satoshi's Genesis address - famous example)")

    utxos = extractor.get_address_utxos(demo_address)

    if utxos:
        print(f"\n✅ Found {len(utxos)} UTXOs")

        # Display first few UTXOs
        print("\n📊 SAMPLE UTXOs:")
        for i, utxo in enumerate(utxos[:3], 1):
            print(f"\n  UTXO #{i}:")
            print(f"    Amount: {utxo['value_btc']:.8f} BTC")
            print(f"    Acquired: {utxo['acquisition_date'][:16] if utxo['acquisition_date'] else 'Unknown'}")
            print(f"    Cost Basis: ${utxo['cost_basis_usd']:,.2f}" if utxo['cost_basis_usd'] else "    Cost Basis: Loading...")
            print(f"    Current Value: ${utxo['current_value_usd']:,.2f}" if utxo['current_value_usd'] else "")
            if utxo.get('unrealized_gain_usd'):
                gain = utxo['unrealized_gain_usd']
                print(f"    Unrealized {'Gain' if gain > 0 else 'Loss'}: ${abs(gain):,.2f}")

        # Calculate cost basis methods
        print("\n\n📈 COST BASIS COMPARISON:")
        methods = extractor.calculate_cost_basis_methods(utxos)

        for method_name, data in methods.items():
            if method_name in ['FIFO', 'LIFO', 'HIFO']:
                print(f"\n  {method_name}:")
                print(f"    Total Cost Basis: ${data['cost_basis_total']:,.2f}")
                print(f"    Per BTC: ${data['cost_basis_per_btc']:,.2f}")

        # Simulate a spend
        print("\n\n💰 SPEND SIMULATION (Selling 0.5 BTC):")
        for method in ['FIFO', 'LIFO', 'HIFO']:
            result = extractor.simulate_spend(utxos, 0.5, method)
            if 'error' not in result:
                print(f"\n  {method}:")
                print(f"    Gain/Loss: ${result['gain_loss_usd']:,.2f}")
                print(f"    Taxable: {'Yes' if result['taxable_event'] else 'No'}")
                print(f"    UTXOs used: {result['utxos_spent']}")

        # Export
        extractor.export_accounting_report(utxos)

    else:
        print("\n⚠️  No UTXOs found or API error")

    print("\n" + "=" * 80)
    print("NEXT STEPS:")
    print("  1. Replace demo_address with your actual Bitcoin wallet address")
    print("  2. Run for full UTXO")